In [25]:
import pandas as pd

drug_prices = pd.read_csv("Drug-Prices-Medicare.csv")
drug_prices.head()

,HCPCS Code,Short Description,HCPCS Code Dosage,Payment Limit,Co-insurance Percentage,Vaccine AWP%,Vaccine Limit,Blood AWP%,Blood limit,Clotting Factor,Notes
0,90371,Hep b ig im,1 ML,137.828,20.0,NaN,NaN,NaN,NaN,NaN,NaN
1,90375,Rabies ig im/sc,150 IU,279.949,20.0,NaN,NaN,NaN,NaN,NaN,NaN
2,90377,Rabies ig ht&sol human im/sc,150 IU,240.133,20.0,NaN,NaN,NaN,NaN,NaN,NaN
3,90586,Bcg vaccine intravesical,1 EACH,163.971,20.0,NaN,NaN,NaN,NaN,NaN,NaN
4,90632,Hepa vaccine adult im,1 ML,73.542,20.0,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
import boto3
import re
import json
session = boto3.Session(profile_name = "aidev", region_name = "us-east-1")
bedrock_client_2 = session.client(service_name="bedrock-runtime", region_name = "us-east-1")

def prompt_with_openai(prompt):
    model_id = "openai.gpt-oss-20b-1:0"

    # Create the request body
    native_request = {
        "model": model_id,  # You can omit this field
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "assistant", "content": "Hello! How can I help you today?"},
            {"role": "user", "content": prompt},
        ],
        "max_completion_tokens": 2000,
        "temperature": 0.4,
        "top_p": 0.6,
        "stream": False,  # You can omit this field
    }

    # Make the InvokeModel request
    response = bedrock_client_2.invoke_model(
        modelId=model_id, body=json.dumps(native_request)
    )

    # Parse and print the message for each choice in the chat completion
    response_body = json.loads(response["body"].read().decode("utf-8"))
    print(response_body)
    for choice in response_body["choices"]:
        response_text = choice["message"]["content"]

    input_tokens = response_body["usage"]["prompt_tokens"]
    print("Input tokens:", input_tokens)
    output_tokens = response_body["usage"]["completion_tokens"]
    print("Output tokens: ", output_tokens)
    token_usage = {
        model_id: {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
        }
    }
    return response_text, token_usage

def extract_and_eval_json(text: str):
    """
    Extracts a substring that starts with { and ends with } from the given text
    and evaluates it as JSON.
    """
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if match:
        try:
            return eval(match.group())
        except json.JSONDecodeError as e:
            print(f"JSON decoding error: {e}")
            return {}
    return {}

json_example = {
    "dosage_value": 1,
    "dosage_unit": "ml"
}

input_text = ""
prompt = lambda input_text, json_example: f"""
I am going to give you HCPCS Dosage, you have to extract the following from it:
dosage_value: integer/float value specified
dosage_unit: unit specified (mg, ml, etc.)

Now, if it is not a valid dosage or not given in units, keep both of them an None
If there are two dosage values given of the same units, give the higher dosage value
If there are multiple dosage values seperated with '-' give the last dosage value mentioned

Finally, give the output in json in the following format:
{json_example}

Here is the HCPCS Dosage:
{input_text}
"""

In [26]:
input_text = "4 MG-4 MG-2 MG"
final_prompt = prompt(input_text, json_example)
result, token_usage = prompt_with_openai(final_prompt)
extract_and_eval_json(result)

{'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': '<reasoning>We need to parse the dosage string: "4 MG-4 MG-2 MG". The instruction: If there are two dosage values given of the same units, give the higher dosage value. If multiple dosage values separated with \'-\' give the last dosage value mentioned. But we have hyphens between values: "4 MG-4 MG-2 MG". This looks like three values: 4 MG, 4 MG, 2 MG. They are separated by hyphens. The rule: If multiple dosage values separated with \'-\' give the last dosage value mentioned. So we should pick the last dosage value: 2 MG. But also rule: If there are two dosage values given of the same units, give the higher dosage value. But we have three values. The last is 2 MG. But the rule about higher dosage might override? The instruction order: "If there are two dosage values given of the same units, give the higher dosage value". But we have three. Maybe we should consider all values and pick the highes

{'dosage_value': 2, 'dosage_unit': 'MG'}

In [ ]:
import pandas as pd
import boto3
import json
import re
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# 1. Setup AWS and Bedrock
session = boto3.Session(profile_name="aidev", region_name="us-east-1")
bedrock_client_2 = session.client(service_name="bedrock-runtime", region_name="us-east-1")

def prompt_with_openai(prompt):
    """Calls the OpenAI model on Bedrock."""
    model_id = "openai.gpt-oss-20b-1:0"
    native_request = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "assistant", "content": "Hello! How can I help you today?"},
            {"role": "user", "content": prompt},
        ],
        "max_completion_tokens": 1000,
        "temperature": 0.4,
        "top_p": 0.6,
        "stream": False,
    }
    try:
        response = bedrock_client_2.invoke_model(
            modelId=model_id, body=json.dumps(native_request)
        )
        response_body = json.loads(response["body"].read().decode("utf-8"))
        return response_body["choices"][0]["message"]["content"]
    except Exception as e:
        return f"Error: {e}"

def extract_and_eval_json(text: str):
    """Extracts JSON from model reasoning output."""
    # Remove reasoning tags if present
    text = text.split("</reasoning>")[-1]
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if match:
        try:
            # Safely handle the string as a dictionary
            # Using json.loads with some cleaning or eval for flexible parsing
            json_str = match.group().replace("'", '"')
            return json.loads(json_str)
        except:
            try: return eval(match.group())
            except: return {"dosage_value": None, "dosage_unit": None}
    return {"dosage_value": None, "dosage_unit": None}

def get_dosage_prompt(input_text):
    """Generates the prompt for a specific dosage string."""
    return f"""
I am going to give you HCPCS Dosage, you have to extract the following from it:
dosage_value: integer/float value specified
dosage_unit: unit specified (mg, ml, etc.)

Rules:
1. If it is not a valid dosage or not given in units, keep both as null.
2. If there are two dosage values given of the same units, give the higher dosage value.
3. If there are multiple dosage values separated with '-' give the last dosage value mentioned.

Output format (JSON only):
{{"dosage_value": 1.0, "dosage_unit": "ML"}}

Here is the HCPCS Dosage:
{input_text}
"""

def process_row(dosage_text):
    """Worker function for parallel execution."""
    if pd.isna(dosage_text) or not str(dosage_text).strip():
        return {"dosage_value": None, "dosage_unit": None}
    
    response = prompt_with_openai(get_dosage_prompt(str(dosage_text)))
    return extract_and_eval_json(response)

# 2. Load and Process
df = pd.read_csv("Drug-Prices-Medicare.csv")
print(f"🚀 Starting parallel processing for {len(df)} rows...")

# Using 10 workers to balance speed and rate limits
with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(tqdm(executor.map(process_row, df["HCPCS Code Dosage"]), total=len(df)))

# 3. Append Results and Save
df["dosage_value"] = [r.get("dosage_value") for r in results]
df["dosage_unit"] = [r.get("dosage_unit") for r in results]

output_file = "Drug-Prices-Medicare-Updated.csv"
df.to_csv(output_file, index=False)

print(f"\n✅ Processing Complete!")
print(f"📂 Updated file saved to: {output_file}")

# Show a sample of the new columns
df[["HCPCS Code Dosage", "dosage_value", "dosage_unit"]].head(10)

In [28]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def process_row(dosage_text):
    """Worker function for parallel execution."""
    if pd.isna(dosage_text) or not str(dosage_text).strip():
        return {"dosage_value": None, "dosage_unit": None}
    
    response, token_usage = prompt_with_openai(prompt(str(dosage_text), json_example))
    return extract_and_eval_json(response)

# 2. Load and Process
df = pd.read_csv("Drug-Prices-Medicare.csv")
print(f"🚀 Starting parallel processing for {len(df)} rows...")

# Using 10 workers to balance speed and rate limits
with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(tqdm(executor.map(process_row, df["HCPCS Code Dosage"]), total=len(df)))

# 3. Append Results and Save
df["dosage_value"] = [r.get("dosage_value") for r in results]
df["dosage_unit"] = [r.get("dosage_unit") for r in results]

output_file = "Drug-Prices-Medicare-Updated.csv"
df.to_csv(output_file, index=False)

print(f"\n✅ Processing Complete!")
print(f"📂 Updated file saved to: {output_file}")

# Show a sample of the new columns
df[["HCPCS Code Dosage", "dosage_value", "dosage_unit"]].head(10)

🚀 Starting parallel processing for 877 rows...


  0%|          | 0/877 [00:00<?, ?it/s]

{'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': '<reasoning>We need to parse "150 IU". The dosage_value: 150, dosage_unit: IU. It\'s a valid dosage. Output JSON. Ensure integer. So {\'dosage_value\': 150, \'dosage_unit\': \'IU\'}.</reasoning>{\'dosage_value\': 150, \'dosage_unit\': \'IU\'}', 'refusal': None, 'role': 'assistant'}}], 'created': 1770029042, 'id': 'chatcmpl-e448d621-4e8c-471e-8265-1bae2d1c0ecf', 'model': 'openai.gpt-oss-20b-1:0', 'object': 'chat.completion', 'service_tier': 'default', 'usage': {'completion_tokens': 75, 'prompt_tokens': 230, 'total_tokens': 305}}
Input tokens: 230
Output tokens:  75
{'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': '<reasoning>We need to parse "150 IU". The dosage_value: 150, dosage_unit: IU. It\'s a valid dosage. Output JSON. Use integer. So {\'dosage_value\': 150, \'dosage_unit\': \'IU\'}.</reasoning>{\'dosage_value\': 150, \'dosage_unit\': \'IU\'}', 're

  0%|          | 1/877 [00:02<31:46,  2.18s/it]

{'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': '<reasoning>We need to parse "0.25 ML". The dosage_value: 0.25, dosage_unit: "ML" (case-insensitive). Output JSON: {\'dosage_value\': 0.25, \'dosage_unit\': \'ml\'}? The example shows unit in lowercase. They used \'ml\' in example. So we should output unit in lowercase. Provide JSON.</reasoning>{\'dosage_value\': 0.25, \'dosage_unit\': \'ml\'}', 'refusal': None, 'role': 'assistant'}}], 'created': 1770029043, 'id': 'chatcmpl-a0a28e5b-4d44-406a-922f-27fe760d3bcd', 'model': 'openai.gpt-oss-20b-1:0', 'object': 'chat.completion', 'service_tier': 'default', 'usage': {'completion_tokens': 106, 'prompt_tokens': 232, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 208}, 'total_tokens': 338}}
Input tokens: 232
Output tokens:  106
{'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': '<reasoning>We need to parse the HCPCS dosage string: "1 EACH". The task

...

100%|█████████▉| 873/877 [01:50<00:00,  7.79it/s]

{'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': '<reasoning>We need to parse the dosage string: "Less than or equal to 100 MG". We need to extract dosage_value: integer/float value specified, dosage_unit: unit specified. If not valid dosage or not given in units, keep both None. If two dosage values same units, give higher. If multiple separated with \'-\', give last. Here we have "100 MG". So dosage_value=100, dosage_unit=\'MG\' (case? maybe \'mg\'). The example output uses \'ml\' lowercase. Should we output unit in lowercase? The example shows \'ml\'. So we should output \'mg\'. So JSON: {\'dosage_value\': 100, \'dosage_unit\': \'mg\'}. Ensure JSON format. Use single quotes? They used single quotes. We\'ll output exactly that.</reasoning>{\'dosage_value\': 100, \'dosage_unit\': \'mg\'}', 'refusal': None, 'role': 'assistant'}}], 'created': 1770029152, 'id': 'chatcmpl-18403907-f6ac-49e1-874d-c7f7cdb24750', 'model': 'openai.gpt-oss-20b-1:0', '

100%|██████████| 877/877 [01:51<00:00,  7.86it/s]

{'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': '<reasoning>We need to parse the dosage. The input: "Greater than 100 MG". The dosage value: 100? But it\'s "Greater than 100 MG". The instruction: If not a valid dosage or not given in units, keep both None. But here we have a unit MG. The dosage value is 100. But "Greater than" indicates maybe not a specific dosage. But we still have a numeric value. The instruction: "If there are two dosage values given of the same units, give the higher dosage value". Not relevant. "If there are multiple dosage values separated with \'-\' give the last dosage value mentioned". Not relevant. So we have a single dosage value 100 mg. Should we treat "Greater than 100 MG" as a valid dosage? It\'s a dosage but not exact. But we have a numeric value. The instruction says: "If it is not a valid dosage or not given in units, keep both of them an None". But we have units. So we can extract 100 mg. So output: {\'dosag

,HCPCS Code Dosage,dosage_value,dosage_unit
0,1 ML,1.00,ml
1,150 IU,150.00,IU
2,150 IU,150.00,IU
3,1 EACH,NaN,None
4,1 ML,1.00,ml
5,0.5 ML,0.50,ml
6,0.5 ML,0.50,ml
7,0.25 ML,0.25,ml
8,0.5 ML,0.50,ml
9,0.2 ML,0.20,ml


In [37]:
updated_df = pd.read_csv("Drug-Prices-Medicare-Updated.csv")

updated_df["dosage_unit"] = updated_df["dosage_unit"].astype(str)
updated_df["dosage_unit"] = updated_df["dosage_unit"].map(lambda x: x.upper())


In [38]:
updated_df["dosage_unit"].unique()

array(['ML', 'IU', 'NAN', 'MCG', 'MG', 'MICROCURIE', 'GM', 'UNITS',
       'UNIT', 'CC', 'MEQ', 'SQ CM', 'PFU'], dtype=object)

In [39]:
updated_df.to_csv("Drug-Prices-Medicare-Updated.csv", index=False)

In [ ]:
import boto3
import json

# --- Configuration ---
# Define the model ID. The specific ID can be found in the AWS Bedrock console.
MODEL_ID = "moonshot.kimi-k2-thinking" 
# Specify your desired AWS region (e.g., 'us-east-1', 'ap-northeast-1')
REGION = "us-east-1" 

session = boto3.Session(profile_name = "aidev", region_name = "us-east-1")
# Set up the Amazon Bedrock runtime client
try:
    bedrock_runtime = session.client(
        service_name="bedrock-runtime",
        region_name=REGION,
    )
except Exception as e:
    print(f"Error connecting to Bedrock: {e}")
    exit()

def invoke_kimi_model(prompt_text):
    """
    Invokes the Kimi K2 Thinking model on Amazon Bedrock.
    """
    MODEL_ID = "moonshot.kimi-k2-thinking" 
    REGION = "us-east-1"
    # The payload structure is specific to the Moonshot API within Bedrock
    payload = {
        "messages": [
            {"role": "system", "content": "You are Kimi, an AI assistant created by Moonshot AI."},
            {"role": "user", "content": prompt_text}
        ],
        "temperature": 1.0, # Recommended for optimal reasoning performance
        "max_tokens": 4096 # Ensure adequate space for full reasoning chains
    }

    body = json.dumps(payload)
    content_type = "application/json"
    accept = "application/json"

    try:
        response = bedrock_runtime.invoke_model(
            body=body,
            modelId=MODEL_ID,
            accept=accept,
            contentType=content_type
        )
        # print(response)
        result_in_byte = (response.get('body').read())
        result = json.loads(str(result_in_byte, encoding='utf-8'))
        # print(result)
        # Extract the generated text
        generated_text = result["choices"][0]["message"]["content"]
        input_tokens = result['usage']['prompt_tokens']
        output_tokens = result['usage']['completion_tokens']
        return generated_text, input_tokens, output_tokens

    except Exception as e:
        print(f"Error invoking model: {e}")
        return None

# --- Example Usage ---
user_question = "Explain the concept of quantum entanglement in simple terms."
response, input_tokens, output_tokens = invoke_kimi_model(user_question)

if response:
    print(f"Kimi Response:\n{response}")
    print(f"Input Tokens: ", input_tokens)
    print(f"Output Tokens: ", output_tokens)


Kimi Response:
 <think> The user asks for a simple explanation of quantum entanglement. I need to break this down into understandable concepts without getting too technical, but still being accurate.

Key points to cover:
1. What it is: particles linked in a special way
2. The key property: measuring one instantly affects the other, no matter the distance
3. The "spooky" part: Einstein's description and why it's weird
4. Common analogies (but also why they fail)
5. No communication faster than light (important clarification)
6. Real-world applications

I should avoid:
- Complex math
- Too much jargon
- Misleading statements about faster-than-light communication
- Oversimplifying to the point of being wrong

Structure:
- Start with a simple analogy (like magic coins or gloves)
- Explain what makes it different from classical correlation
- Address the distance independence
- Clarify the measurement aspect
- Mention the no-communication theorem simply
- Give practical examples/application

## Medical Necessity Flag

In [1]:
import pandas as pd 

data = pd.read_csv("cache/dataframe/ae52691527d1858df32f93975ceaa2be.csv")
data.head()

,Document Name,Date,Medical Procedure Name,CPT Code,CPT Code Source,Modifier,Quantity,Unit Price,Total Price,Entity Type,...,Dosage Unit,Units from Name,Page,Extraction Confidence,Unit Price Threshold,No. of Medicare Units,Total Threshold Price,Threshold Multiplier,Price Difference,Highlight Geometry
0,GP2.pdf,2022-08-05,XR CLAVICLE LT,73000,extracted,LT,1,200.00,200.00,procedure,...,NaN,NaN,3,95,25.38,1.0,25.38,7.88,174.62,"{""Width"": 0.355433851480484, ""Height"": 0.01183..."
1,GP2.pdf,2022-08-05,XR SHOULDER 2V/MORE,73030,extracted,LT,1,220.00,220.00,procedure,...,NaN,NaN,3,95,26.72,1.0,26.72,8.23,193.28,"{""Width"": 0.25812920928001404, ""Height"": 0.011..."
2,GP2.pdf,2022-08-05,DEXAMETHASONE 4MG/ML,J1100,inferred,NaN,4,7.92,31.68,drug,...,MG,1.0,3,90,0.10,16.0,1.68,18.86,30.00,"{""Width"": 0.2683259844779968, ""Height"": 0.0114..."
3,GP2.pdf,2022-08-05,KETOROLAC 60MG VIAL,J1885,inferred,NaN,4,12.00,48.00,drug,...,MG,1.0,3,90,0.33,16.0,5.33,9.01,42.67,"{""Width"": 0.2581912577152252, ""Height"": 0.0096..."
4,GP2.pdf,2022-08-05,METHOCARBAMOL 500MG,J2800,inferred,NaN,2,6.00,12.00,drug,...,MG,1.0,3,85,4.54,2.0,9.07,1.32,2.93,"{""Width"": 0.25746843218803406, ""Height"": 0.009..."


In [2]:
data["Medical Procedure Name"].unique()

array(['XR CLAVICLE LT', 'XR SHOULDER 2V/MORE', 'DEXAMETHASONE 4MG/ML',
       'KETOROLAC 60MG VIAL', 'METHOCARBAMOL 500MG', 'ER LEVEL FOUR - 25',
       'IM INJECTION', 'PR EST EVAL VISIT LVL 3',
       'MRI joint upper extremity w/o dye', 'ALS Emergency Transport',
       'Mileage', 'PT EVAL: LOW COMPLEX 20', 'Therapeutic Exercise',
       'Electrical Stimulation - Not', 'Neuromuscular Re-ed'],
      dtype=object)

In [27]:
import re
import json
from pathlib import Path
from typing import Dict, List, Any, Optional
import PyPDF2
import pdfplumber
from dataclasses import dataclass, asdict

# pip install PyPDF2 pdfplumber


class PDFExtractor:
    """Extract text from PDF files"""
    
    def __init__(self, pdf_path: str):
        self.pdf_path = Path(pdf_path)
        self.text = ""
        
    def extract_text(self, method: str = 'pdfplumber') -> str:
        """Extract text from PDF using specified method"""
        if method == 'pdfplumber':
            try:
                text = ""
                with pdfplumber.open(self.pdf_path) as pdf:
                    for page in pdf.pages:
                        page_text = page.extract_text()
                        if page_text:
                            text += page_text + "\n"
                self.text = text
            except Exception as e:
                print(f"pdfplumber extraction failed: {e}")
                # Fallback to PyPDF2
                method = 'pypdf2'
        
        if method == 'pypdf2':
            try:
                text = ""
                with open(self.pdf_path, 'rb') as file:
                    pdf_reader = PyPDF2.PdfReader(file)
                    for page in pdf_reader.pages:
                        text += page.extract_text() + "\n"
                self.text = text
            except Exception as e:
                print(f"PyPDF2 extraction failed: {e}")
        
        return self.text


@dataclass
class ClaimSummaryData:
    """Data class to store extracted claim information"""
    claim_file: str
    incident_date: Optional[str]
    fnol_date: Optional[str]
    claimant_name: str
    accident_description: str
    accident_related_injuries: List[str]
    preexisting_conditions: List[str]
    body_parts_injured: List[str]
    medical_providers: List[str]
    total_economic_damages: float
    past_medical_expenses: float
    future_medical_expenses: str
    
    def to_dict(self) -> Dict:
        return asdict(self)
    
    def to_json(self) -> str:
        return json.dumps(self.to_dict(), indent=2)


class ClaimSummaryExtractor:
    """Extract relevant information from claim summary PDF"""
    
    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.pdf_extractor = PDFExtractor(pdf_path)
        self.text = ""
        self.data: Optional[ClaimSummaryData] = None
    
    def extract_claim_file(self) -> str:
        """Extract claim file number"""
        patterns = [
            r'Claim file Name/Number\s+([A-Z0-9\-]+)',
            r'Claim File:\s*([A-Z0-9\-]+)',
        ]
        for pattern in patterns:
            match = re.search(pattern, self.text, re.IGNORECASE)
            if match:
                return match.group(1).strip()
        return "UNKNOWN"
    
    def extract_dates(self) -> Dict[str, Optional[str]]:
        """Extract incident date and FNOL date"""
        dates = {'incident_date': None, 'fnol_date': None}
        
        doi_patterns = [
            r'Date of Incident/Loss is (\d{2}-\d{2}-\d{4})',
            r'Date of Loss:\s*(\d{2}-\d{2}-\d{4})',
        ]
        for pattern in doi_patterns:
            match = re.search(pattern, self.text)
            if match:
                dates['incident_date'] = match.group(1)
                break
        
        fnol_patterns = [
            r'Date of First Notice of Loss is (\d{2}-\d{2}-\d{4})',
            r'FNOL:\s*(\d{2}-\d{2}-\d{4})',
        ]
        for pattern in fnol_patterns:
            match = re.search(pattern, self.text)
            if match:
                dates['fnol_date'] = match.group(1)
                break
        
        return dates
    
    def extract_claimant_name(self) -> str:
        """Extract claimant name"""
        patterns = [
            r"Claimant's Name\s+([\w\s]+?)(?:Stephanie|Date of birth|\n\n)",
            r'Claimant:\s*([\w\s]+?)(?:\n|$)',
        ]
        for pattern in patterns:
            match = re.search(pattern, self.text, re.DOTALL)
            if match:
                name = match.group(1).strip()
                name = re.sub(r'\s+', ' ', name)
                return name
        return "UNKNOWN"
    
    def extract_accident_related_injuries(self) -> List[str]:
        """Extract injuries identified as accident-related"""
        injuries = []
        patterns = [
            r'Injuries/Medical Conditions Identified for ([^:]+):(.*?)(?=Injuries/Medical conditions Related|Medical conditions or injuries that existed prior|Medical providers)',
        ]
        for pattern in patterns:
            match = re.search(pattern, self.text, re.DOTALL | re.IGNORECASE)
            if match:
                injury_text = match.group(2)
                injury_items = re.findall(r'\d+\.\s+([^\n]+)', injury_text)
                injuries.extend([inj.strip() for inj in injury_items])
                break
        return injuries if injuries else ["Not specified"]
    
    def extract_preexisting_conditions(self) -> List[str]:
        """Extract pre-existing medical conditions"""
        preexisting = []
        patterns = [
            r'Pre-existing medical conditions or injuries for ([^:]+):(.*?)(?=Medical providers that have examined|Medical Providers|$)',
        ]
        for pattern in patterns:
            match = re.search(pattern, self.text, re.DOTALL | re.IGNORECASE)
            if match:
                condition_text = match.group(2)
                condition_items = re.findall(r'\d+\.\s+([^\n]+)', condition_text)
                preexisting.extend([cond.strip() for cond in condition_items])
                break
        return preexisting
    
    def extract_medical_providers(self) -> List[str]:
        """Extract list of medical providers"""
        providers = []
        patterns = [
            r'Medical providers for ([^:]+):(.*?)(?=Reports/Records from medical professionals|Reports/Records|$)',
        ]
        for pattern in patterns:
            match = re.search(pattern, self.text, re.DOTALL | re.IGNORECASE)
            if match:
                provider_text = match.group(2)
                provider_items = re.findall(r'\d+\.\s+([^\n]+)', provider_text)
                providers.extend([prov.strip() for prov in provider_items])
                break
        return providers
    
    def extract_economic_damages(self) -> Dict[str, Any]:
        """Extract medical expenses and economic damages"""
        damages = {
            'total_economic_damages': 0.0,
            'past_medical_expenses': 0.0,
            'future_medical_expenses': 'Not specified'
        }
        
        total_patterns = [r'Total Economic Damages are \$?([\d,]+\.?\d*)']
        for pattern in total_patterns:
            match = re.search(pattern, self.text)
            if match:
                damages['total_economic_damages'] = float(match.group(1).replace(',', ''))
                break
        
        past_patterns = [
            r'Past medical expenses:\s*\$?([\d,]+\.?\d*)',
            r'Total medical expenses:\s*\$?([\d,]+\.?\d*)'
        ]
        for pattern in past_patterns:
            match = re.search(pattern, self.text)
            if match:
                damages['past_medical_expenses'] = float(match.group(1).replace(',', ''))
                break
        
        future_patterns = [r'Future medical expenses:\s*([^\n]+)']
        for pattern in future_patterns:
            match = re.search(pattern, self.text)
            if match:
                damages['future_medical_expenses'] = match.group(1).strip()
                break
        
        return damages
    
    def extract_accident_description(self) -> str:
        """Extract the accident description"""
        patterns = [
            r'Accident or incident Summary\s+(.*?)(?=Location of the accident|Location:|$)',
        ]
        for pattern in patterns:
            match = re.search(pattern, self.text, re.DOTALL | re.IGNORECASE)
            if match:
                description = match.group(1).strip()
                description = re.sub(r'\s+', ' ', description)
                description = re.sub(r'$$\d+$$', '', description)
                return description[:500]
        return "Not specified"
    
    def extract_body_parts_injured(self) -> List[str]:
        """Extract specific body parts mentioned as injured"""
        body_parts = set()
        body_part_keywords = [
            'shoulder', 'arm', 'leg', 'back', 'neck', 'head', 
            'knee', 'ankle', 'wrist', 'hip', 'chest', 'abdomen',
            'spine', 'hand', 'foot', 'elbow', 'ribs', 'pelvis'
        ]
        
        for keyword in body_part_keywords:
            if re.search(rf'\b{keyword}\b', self.text, re.IGNORECASE):
                body_parts.add(keyword.lower())
        
        side_body_parts = re.findall(r'\b(left|right)\s+(\w+)', self.text, re.IGNORECASE)
        for side, part in side_body_parts:
            if part.lower() in body_part_keywords:
                body_parts.add(f"{side.lower()} {part.lower()}")
        
        return list(body_parts) if body_parts else ["Not specified"]
    
    def extract_all(self) -> ClaimSummaryData:
        """Extract all relevant information from PDF"""
        # Extract text from PDF
        self.text = self.pdf_extractor.extract_text()
        
        if not self.text.strip():
            raise ValueError("Failed to extract text from PDF")
        
        # Extract all information
        dates = self.extract_dates()
        damages = self.extract_economic_damages()
        
        self.data = ClaimSummaryData(
            claim_file=self.extract_claim_file(),
            incident_date=dates.get('incident_date'),
            fnol_date=dates.get('fnol_date'),
            claimant_name=self.extract_claimant_name(),
            accident_description=self.extract_accident_description(),
            accident_related_injuries=self.extract_accident_related_injuries(),
            preexisting_conditions=self.extract_preexisting_conditions(),
            body_parts_injured=self.extract_body_parts_injured(),
            medical_providers=self.extract_medical_providers(),
            total_economic_damages=damages['total_economic_damages'],
            past_medical_expenses=damages['past_medical_expenses'],
            future_medical_expenses=damages['future_medical_expenses']
        )
        
        return self.data
    
    def get_bill_review_context(self) -> Dict[str, Any]:
        """Get formatted context for medical bill review"""
        if not self.data:
            self.extract_all()
        
        return {
            'claim_file': self.data.claim_file,
            'incident_date': self.data.incident_date,
            'claimant_name': self.data.claimant_name,
            'acceptable_injuries': self.data.accident_related_injuries,
            'excluded_conditions': self.data.preexisting_conditions,
            'affected_body_parts': self.data.body_parts_injured,
            'authorized_providers': self.data.medical_providers,
            'total_approved_amount': self.data.past_medical_expenses,
            'accident_summary': self.data.accident_description
        }


def extract_claim_summary(pdf_path: str, output_json_path: str = None) -> Dict[str, Any]:
    """
    Main function to extract claim summary from PDF
    
    Args:
        pdf_path: Path to claim summary PDF
        output_json_path: Optional path to save JSON output
    
    Returns:
        Dictionary with extracted information
    """
    extractor = ClaimSummaryExtractor(pdf_path)
    claim_data = extractor.extract_all()
    
    # Get both formats
    full_data = claim_data.to_dict()
    bill_review_context = extractor.get_bill_review_context()
    
    result = {
        'full_claim_data': full_data,
        'bill_review_context': bill_review_context
    }
    
    # Save to JSON if path provided
    if output_json_path:
        with open(output_json_path, 'w') as f:
            json.dump(result, f, indent=2)
        print(f"✅ Results saved to: {output_json_path}")
    
    return result


# Example usage
if __name__ == "__main__":
    # Replace with your PDF path
    pdf_path = "./fake_files/GP-File-2-claim_summary.pdf"
    
    # Extract claim summary
    result = extract_claim_summary(pdf_path, "./fake_files/extracted_claim_info.json")
    
    # Print results
    print("\n" + "="*70)
    print("EXTRACTED CLAIM INFORMATION")
    print("="*70)
    print(json.dumps(result, indent=2))


# # Simple extraction

# result = extract_claim_summary("fask", "output.json")

# # Access data
# claim_info = result['full_claim_data']
# bill_context = result['bill_review_context']

# print(f"Claim File: {claim_info['claim_file']}")
# print(f"Incident Date: {claim_info['incident_date']}")
# print(f"Injuries: {claim_info['accident_related_injuries']}")

✅ Results saved to: ./fake_files/extracted_claim_info.json

EXTRACTED CLAIM INFORMATION
{
  "full_claim_data": {
    "claim_file": "GP-File-2",
    "incident_date": "08-05-2022",
    "fnol_date": "08-05-2022",
    "claimant_name": "Roger Pouncey",
    "accident_description": "On August 5, 2022, at approximately 20:14 (8:14 PM), a motor vehicle collision occurred when another vehicle crossed the centerline and struck the patient's vehicle on the driver's side [26][27][28]. The patient, who was driving and restrained at the time, experienced a T-bone impact that did not trigger airbag deployment and did not result in loss of consciousness [26][28]. Following the collision, the patient suffered left shoulder pain and restricted movement, and was subsequently transported",
    "accident_related_injuries": [
      "Acute pain in left shoulder",
      "Motor vehicle accident injury (as a restrained driver)",
      "Unspecified motor vehicle accident-related injury"
    ],
    "preexisting_co

In [9]:
import re
import json
from pathlib import Path
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, asdict
import PyPDF2
import pdfplumber

# pip install PyPDF2 pdfplumber


class PDFExtractor:
    """Extract text from PDF files"""
    
    def __init__(self, pdf_path: str):
        self.pdf_path = Path(pdf_path)
        self.text = ""
        
    def extract_text(self, method: str = 'pdfplumber') -> str:
        """Extract text from PDF"""
        try:
            if method == 'pdfplumber':
                text = ""
                with pdfplumber.open(self.pdf_path) as pdf:
                    for page in pdf.pages:
                        page_text = page.extract_text()
                        if page_text:
                            text += page_text + "\n"
                self.text = text
            else:
                text = ""
                with open(self.pdf_path, 'rb') as file:
                    pdf_reader = PyPDF2.PdfReader(file)
                    for page in pdf_reader.pages:
                        text += page.extract_text() + "\n"
                self.text = text
        except Exception as e:
            print(f"Extraction failed: {e}")
        
        return self.text


@dataclass
class MedicalSummaryData:
    """Extracted medical summary for bill review"""
    claim_file: str
    incident_date: str
    
    # Treatment timeline
    pre_accident_treatments: List[Dict[str, str]]  # Date + condition treated
    post_accident_treatments: List[Dict[str, str]]  # Date + condition treated
    
    # Medical conditions
    accident_related_conditions: List[str]
    preexisting_conditions: List[str]
    
    # Providers
    treating_providers: List[str]
    
    # Medications
    accident_related_medications: List[str]
    preexisting_medications: List[str]
    
    def to_dict(self) -> Dict:
        return asdict(self)
    
    def to_json(self) -> str:
        return json.dumps(self.to_dict(), indent=2)


class MedicalSummaryExtractor:
    """Extract relevant information from medical summary PDF for bill review"""
    
    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.pdf_extractor = PDFExtractor(pdf_path)
        self.text = ""
        self.data: Optional[MedicalSummaryData] = None
    
    def extract_claim_file(self) -> str:
        """Extract claim file number"""
        match = re.search(r'Claim File Name:\s*([A-Z0-9\-]+)', self.text)
        return match.group(1) if match else "UNKNOWN"
    
    def extract_incident_date(self) -> str:
        """Extract incident date"""
        patterns = [
            r'Date of Loss or Date of Incident:\s*(\d{2}/\d{2}/\d{4})',
            r'Incident Date:\s*(\d{2}/\d{2}/\d{4})'
        ]
        for pattern in patterns:
            match = re.search(pattern, self.text)
            if match:
                return match.group(1)
        return "UNKNOWN"
    
    def extract_treatments_by_period(self) -> tuple:
        """Extract treatments before and after incident"""
        pre_accident = []
        post_accident = []
        
        # Find "Before incident" section
        before_match = re.search(
            r'Before incident\s+Date\s+Abstract(.*?)After incident',
            self.text,
            re.DOTALL
        )
        
        if before_match:
            before_text = before_match.group(1)
            # Extract date entries
            date_entries = re.findall(
                r'(\d{2}/\d{2}/\d{4})\s+(.*?)(?=\d{2}/\d{2}/\d{4}|Providers Involved)',
                before_text,
                re.DOTALL
            )
            
            for date, description in date_entries:
                # Extract key conditions
                conditions = self._extract_conditions_from_text(description)
                if conditions:
                    pre_accident.append({
                        'date': date,
                        'conditions': ', '.join(conditions[:3])  # Limit to 3
                    })
        
        # Find "After incident" section
        after_match = re.search(
            r'After incident\s+Date\s+Abstract(.*?)(?=1\.3 ICD Groups|$)',
            self.text,
            re.DOTALL
        )
        
        if after_match:
            after_text = after_match.group(1)
            date_entries = re.findall(
                r'(\d{2}/\d{2}/\d{4})\s+(.*?)(?=\d{2}/\d{2}/\d{4}|Providers Involved)',
                after_text,
                re.DOTALL
            )
            
            for date, description in date_entries:
                conditions = self._extract_conditions_from_text(description)
                if conditions:
                    post_accident.append({
                        'date': date,
                        'conditions': ', '.join(conditions[:3])
                    })
        
        return pre_accident, post_accident
    
    def _extract_conditions_from_text(self, text: str) -> List[str]:
        """Extract medical conditions from description text"""
        conditions = []
        
        # Look for condition patterns
        condition_patterns = [
            r'([A-Z][a-z]+ [A-Z][a-z]+ [A-Z][a-z]+)$$\d+$$',  # Multi-word conditions
            r'([A-Z][a-z]+ [A-Z][a-z]+)$$\d+$$',  # Two-word conditions
            r'([A-Z][a-z]+)$$\d+$$'  # Single-word conditions
        ]
        
        for pattern in condition_patterns:
            matches = re.findall(pattern, text)
            conditions.extend(matches)
        
        # Remove duplicates while preserving order
        seen = set()
        unique_conditions = []
        for cond in conditions:
            if cond.lower() not in seen:
                seen.add(cond.lower())
                unique_conditions.append(cond)
        
        return unique_conditions[:5]  # Max 5 conditions
    
    def extract_accident_related_conditions(self) -> List[str]:
        """Extract accident-related medical conditions"""
        conditions = []
        
        # Look in ICD Groups for "Cause" or "Accident" type
        icd_match = re.search(
            r'ICD Group\s+Medical conditions\s+Condition type(.*?)(?=1\.4 Medical Necessity|$)',
            self.text,
            re.DOTALL
        )
        
        if icd_match:
            icd_text = icd_match.group(1)
            
            # Extract conditions marked as Accident or Cause
            accident_lines = re.findall(
                r'([A-Z]\d+(?:\.\d+)?[A-Z]? - [^\n]+)\s+(?:Accident|Cause)',
                icd_text
            )
            
            conditions.extend(accident_lines)
        
        # Deduplicate and clean
        return list(set(conditions))[:10]  # Max 10
    
    def extract_preexisting_conditions(self) -> List[str]:
        """Extract pre-existing conditions"""
        conditions = []
        
        icd_match = re.search(
            r'ICD Group\s+Medical conditions\s+Condition type(.*?)(?=1\.4 Medical Necessity|$)',
            self.text,
            re.DOTALL
        )
        
        if icd_match:
            icd_text = icd_match.group(1)
            
            # Extract conditions marked as Pre-existing or Comorbidity
            preexisting_lines = re.findall(
                r'([A-Z]\d+(?:\.\d+)?[A-Z]? - [^\n]+)\s+(?:Pre-existing Condition|Comorbidity)',
                icd_text
            )
            
            conditions.extend(preexisting_lines)
        
        return list(set(conditions))[:10]
    
    def extract_treating_providers(self) -> List[str]:
        """Extract unique treating providers"""
        providers = set()
        
        # Extract from "Providers Involved" mentions
        provider_matches = re.findall(
            r'Providers Involved:\s*([^\n]+)',
            self.text
        )
        
        for match in provider_matches:
            # Split by comma and clean
            provider_list = [p.strip() for p in match.split(',')]
            providers.update(provider_list)
        
        # Clean up common non-provider entries
        exclude_words = ['ChartPro Incorporated', 'Incorporated']
        providers = {
            p for p in providers 
            if p and not any(excl in p for excl in exclude_words)
        }
        
        return sorted(list(providers))[:15]  # Max 15
    
    def extract_medications(self) -> tuple:
        """Extract accident-related vs pre-existing medications"""
        accident_meds = []
        preexisting_meds = []
        
        # Find medication list section
        med_match = re.search(
            r'1\.5 Medication List(.*?)(?=Summary generated|$)',
            self.text,
            re.DOTALL
        )
        
        if med_match:
            med_text = med_match.group(1)
            
            # Extract medications from table
            med_entries = re.findall(
                r'([A-Za-z]+(?:-[A-Za-z]+)?)\s+\d+\s*(?:mg|grams|Units|tablet)',
                med_text,
                re.IGNORECASE
            )
            
            # Categorize based on common pain/injury vs chronic meds
            pain_keywords = ['ketorolac', 'methocarbamol', 'ibuprofen', 
                           'prednisone', 'robaxin', 'naproxen', 'tizanidine', 
                           'zanaflex', 'dexamethasone']
            
            for med in set(med_entries):
                med_lower = med.lower()
                if any(keyword in med_lower for keyword in pain_keywords):
                    accident_meds.append(med)
                else:
                    preexisting_meds.append(med)
        
        return accident_meds[:10], preexisting_meds[:10]
    
    def extract_all(self) -> MedicalSummaryData:
        """Extract all relevant information"""
        self.text = self.pdf_extractor.extract_text()
        
        if not self.text.strip():
            raise ValueError("Failed to extract text from PDF")
        
        pre_treatments, post_treatments = self.extract_treatments_by_period()
        accident_meds, preexisting_meds = self.extract_medications()
        
        self.data = MedicalSummaryData(
            claim_file=self.extract_claim_file(),
            incident_date=self.extract_incident_date(),
            pre_accident_treatments=pre_treatments,
            post_accident_treatments=post_treatments,
            accident_related_conditions=self.extract_accident_related_conditions(),
            preexisting_conditions=self.extract_preexisting_conditions(),
            treating_providers=self.extract_treating_providers(),
            accident_related_medications=accident_meds,
            preexisting_medications=preexisting_meds
        )
        
        return self.data
    
    def get_bill_review_context(self) -> Dict[str, Any]:
        """Get concise context for bill review"""
        if not self.data:
            self.extract_all()
        
        return {
            'claim_file': self.data.claim_file,
            'incident_date': self.data.incident_date,
            'pre_accident_treatment_dates': [t['date'] for t in self.data.pre_accident_treatments],
            'post_accident_treatment_dates': [t['date'] for t in self.data.post_accident_treatments],
            'accident_conditions': self.data.accident_related_conditions,
            'preexisting_conditions': self.data.preexisting_conditions,
            'authorized_providers': self.data.treating_providers,
            'accident_medications': self.data.accident_related_medications
        }


def extract_medical_summary(pdf_path: str, output_json_path: str = None) -> Dict[str, Any]:
    """
    Main function to extract medical summary from PDF
    
    Args:
        pdf_path: Path to medical summary PDF
        output_json_path: Optional path to save JSON output
    
    Returns:
        Dictionary with extracted information
    """
    extractor = MedicalSummaryExtractor(pdf_path)
    medical_data = extractor.extract_all()
    
    full_data = medical_data.to_dict()
    bill_context = extractor.get_bill_review_context()
    
    result = {
        'full_medical_data': full_data,
        'bill_review_context': bill_context
    }
    
    if output_json_path:
        with open(output_json_path, 'w') as f:
            json.dump(result, f, indent=2)
        print(f"✅ Results saved to: {output_json_path}")
    
    return result


# Example usage
if __name__ == "__main__":
    pdf_path = "./fake_files/GP-File-2-chronological_medical_summary_file.pdf"
    
    result = extract_medical_summary(pdf_path, "./fake_files/extracted_medical_summary.json")
    
    print("\n" + "="*70)
    print("EXTRACTED MEDICAL SUMMARY (BILL REVIEW CONTEXT)")
    print("="*70)
    print(json.dumps(result['bill_review_context'], indent=2))

✅ Results saved to: ./fake_files/extracted_medical_summary.json

EXTRACTED MEDICAL SUMMARY (BILL REVIEW CONTEXT)
{
  "claim_file": "GP-F",
  "incident_date": "08/05/2022",
  "pre_accident_treatment_dates": [],
  "post_accident_treatment_dates": [],
  "accident_conditions": [
    "T85 - Electrical Stimulation",
    "M25 - Pain In Left Shoulder",
    "Z04 - Accident",
    "M25.512 - Pain In Left Shoulder",
    "V87 - Motor Vehicle Collison",
    "S42.002A - Left Clavicle Injury",
    "M25.512 - Pain"
  ],
  "preexisting_conditions": [
    "F40 - Claustrophobic",
    "Z98.2 - Cerebrospinal Fluid Drainage Device",
    "I63.9 - Stroke",
    "F33 - Mdd",
    "G47 - Obstructive Sleep Apnea Of Fluid",
    "I10 - Hypertension",
    "I63 - Cva",
    "F33.1 - Major Depressive Disorder",
    "E29 - Testicular Hypofunction",
    "E66 - Obesity"
  ],
  "authorized_providers": [
    "BRIDGEWAY DIAGNOSTIC RADIOLOGY",
    "Cameron",
    "Dennis Harden",
    "Dennis K",
    "Dennis K Harden",
    "Denni

In [10]:
!pip install tabula-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 2.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [14]:
!pip install jpype1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 582.2/582.2 kB 2.8 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [15]:
import PyPDF2
import re
import json
import tabula  # For table extraction (requires: pip install tabula-py)
from collections import defaultdict

def extract_icd_with_tabula(pdf_path):
    """
    Extract ICD codes using tabula for better table parsing
    Requires: pip install tabula-py
    """
    result = {
        "Cause": [],
        "Accident": [],
        "Comorbidity": [],
        "Pre-existing Condition": []
    }
    
    try:
        # Extract tables from PDF
        tables = tabula.read_pdf(pdf_path, pages='all', multiple_tables=True)
        
        for table in tables:
            # Check if this is the ICD Groups table
            if 'ICD Group' in str(table.columns) or 'Medical conditions' in str(table.columns):
                for _, row in table.iterrows():
                    try:
                        # Extract ICD code and description
                        medical_condition = str(row.get('Medical conditions', ''))
                        condition_type = str(row.get('Condition type', ''))
                        
                        # Parse ICD code from medical condition
                        icd_match = re.match(r'^([A-Z]\d+(?:\.\d+)?[A-Z]*)\s*-\s*(.+)$', medical_condition)
                        
                        if icd_match and condition_type in result:
                            entry = {
                                "code": icd_match.group(1).strip(),
                                "description": icd_match.group(2).strip()
                            }
                            result[condition_type].append(entry)
                    except Exception as e:
                        continue
    
    except Exception as e:
        print(f"Tabula extraction failed: {e}")
        print("Falling back to text extraction...")
        return None
    
    return result

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF file"""
    text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            text += page.extract_text()
    return text

def extract_icd_codes_enhanced(pdf_path):
    """
    Enhanced version to extract ICD codes with better parsing
    """
    text = extract_text_from_pdf(pdf_path)
    
    result = {
        "Cause": [],
        "Accident": [],
        "Comorbidity": [],
        "Pre-existing Condition": []
    }
    
    # Find ICD Groups section
    icd_start = text.find("1.3 ICD Groups")
    icd_end = text.find("1.4 Medical Necessity")
    
    if icd_start == -1 or icd_end == -1:
        print("ICD Groups section not found")
        return result
    
    icd_section = text[icd_start:icd_end]
    
    # Split into lines for better parsing
    lines = icd_section.split('\n')
    
    current_group = None
    i = 0
    
    while i < len(lines):
        line = lines[i].strip()
        
        # Skip empty lines and headers
        if not line or line in ["ICD Group", "Medical conditions", "Condition type"]:
            i += 1
            continue
        
        # Check if line contains ICD code pattern
        # Pattern: ICD_CODE - Description
        icd_pattern = r'^([A-Z]\d+(?:\.\d+)?[A-Z]*)\s*-\s*(.+)$'
        match = re.match(icd_pattern, line)
        
        if match:
            icd_code = match.group(1).strip()
            description = match.group(2).strip()
            
            # Look ahead for condition type
            condition_type = None
            if i + 1 < len(lines):
                next_line = lines[i + 1].strip()
                if next_line in ["Cause", "Accident", "Comorbidity", "Pre-existing Condition"]:
                    condition_type = next_line
                    i += 1  # Skip the condition type line
            
            if condition_type:
                entry = {
                    "code": icd_code,
                    "description": description
                }
                result[condition_type].append(entry)
        
        i += 1
    
    return result

def extract_icd_codes_comprehensive(pdf_path):
    """
    Comprehensive extraction trying multiple methods
    """
    # Try tabula first (more accurate for tables)
    try:
        result = extract_icd_with_tabula(pdf_path)
        if result and any(result.values()):
            return result
    except:
        pass
    
    # Fallback to text extraction
    return extract_icd_codes_enhanced(pdf_path)



In [16]:
result = extract_icd_with_tabula("./fake_files/GP-File-2-chronological_medical_summary_file.pdf")

Error from tabula-java:
The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.





Tabula extraction failed: Command '['java', '-Djava.awt.headless=true', '-Dfile.encoding=UTF8', '-jar', '/Users/yeshwanth.penukonda/Desktop/Doclens/Antigravity_testing/MBR_Streamlit/venv/lib/python3.12/site-packages/tabula/tabula-1.0.5-jar-with-dependencies.jar', '--pages', 'all', '--guess', '--format', 'JSON', './fake_files/GP-File-2-chronological_medical_summary_file.pdf']' returned non-zero exit status 1.
Falling back to text extraction...


In [17]:
import json
import re

def extract_icd_groups_from_text(text):
    """
    Extract ICD groups from the medical summary text and categorize by condition type.
    """
    result = {
        "cause": [],
        "accident-related": [],
        "pre-existing condition": [],
        "comorbidity": []
    }

    # Find the ICD Groups section
    icd_section_match = re.search(r'1\.3\s*ICD\s*Groups(.*?)1\.4\s*Medical\s*Necessity', text, re.DOTALL | re.IGNORECASE)

    if not icd_section_match:
        # Try alternative boundary
        icd_section_match = re.search(r'ICD\s*Group\s*Medical\s*conditions\s*Condition\s*type(.*?)1\.4', text, re.DOTALL | re.IGNORECASE)

    if not icd_section_match:
        print("Could not find ICD Groups section in text.")
        return result

    icd_text = icd_section_match.group(1)

    # Pattern to match lines like: "V59.9XXD - Motor Vehicle Accident Injuring Restrained Driver, Subsequent Encounter    Cause"
    # or "D49 - Brain Tumor    Pre-existing Condition"
    # The ICD code and description are separated by " - " and the condition type is at the end

    # We'll match entries that have an ICD code, description, and condition type
    # ICD codes typically look like: V59.9XXD, M25.512, I10, D49, F33.1, etc.
    pattern = re.compile(
        r'([A-Z]\d{1,2}(?:\.\d{1,4})?(?:[A-Z]{0,4}\d{0,2})?)\s*[-–]\s*'  # ICD code
        r'(.+?)\s+'  # Description
        r'(Cause|Accident|Pre-existing\s*Condition|Comorbidity)\s*$',  # Condition type
        re.MULTILINE | re.IGNORECASE
    )

    matches = pattern.findall(icd_text)

    for match in matches:
        icd_code = match[0].strip()
        description = match[1].strip()
        # Clean up trailing commas or whitespace from description
        description = re.sub(r'\s+', ' ', description).strip().rstrip(',')
        condition_type = match[2].strip().lower()

        entry = f"{icd_code} - {description}"

        if condition_type == "cause":
            result["cause"].append(entry)
        elif condition_type == "accident":
            result["accident-related"].append(entry)
        elif "pre-existing" in condition_type:
            result["pre-existing condition"].append(entry)
        elif condition_type == "comorbidity":
            result["comorbidity"].append(entry)

    return result


def extract_from_pdf_pdfplumber(pdf_path):
    """Extract text from PDF using pdfplumber."""
    import pdfplumber

    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"

            # Also try extracting tables
            tables = page.extract_tables()
            for table in tables:
                for row in table:
                    if row:
                        cleaned_row = [str(cell).strip() if cell else "" for cell in row]
                        full_text += " ".join(cleaned_row) + "\n"

    return full_text


def extract_from_pdf_pymupdf(pdf_path):
    """Extract text from PDF using PyMuPDF (fitz)."""
    import fitz  # PyMuPDF

    full_text = ""
    doc = fitz.open(pdf_path)
    for page in doc:
        full_text += page.get_text() + "\n"
    doc.close()

    return full_text


def extract_from_pdf_pdfminer(pdf_path):
    """Extract text from PDF using pdfminer."""
    from pdfminer.high_level import extract_text

    full_text = extract_text(pdf_path)
    return full_text


def parse_icd_from_structured_text(text):
    """
    Alternative parser that handles the table structure more robustly
    by looking for ICD group headers and parsing entries under them.
    """
    result = {
        "cause": [],
        "accident-related": [],
        "pre-existing condition": [],
        "comorbidity": []
    }

    # Normalize whitespace
    lines = text.split('\n')
    cleaned_lines = [line.strip() for line in lines if line.strip()]

    # Find ICD section boundaries
    start_idx = None
    end_idx = None

    for i, line in enumerate(cleaned_lines):
        if 'ICD Group' in line and 'Medical conditions' in line:
            start_idx = i + 1
        elif '1.4' in line and 'Medical Necessity' in line:
            end_idx = i
            break

    if start_idx is None:
        # Try another approach - look for the section header
        for i, line in enumerate(cleaned_lines):
            if '1.3 ICD Groups' in line or '1.3ICD Groups' in line:
                start_idx = i + 1
            elif '1.4 Medical Necessity' in line or '1.4Medical Necessity' in line:
                end_idx = i
                break

    if start_idx is None or end_idx is None:
        return None

    icd_lines = cleaned_lines[start_idx:end_idx]

    # Known ICD group headers (these appear as row headers in the table)
    group_headers = [
        'V00-V99', 'S40-S49', 'M00-M25', 'C00-D89', 'I00-I99',
        'F01-F99', 'E00-E89', 'G00-G99', 'Z00-Z99', 'T66-T88', 'H00-H59'
    ]

    # Combine all lines into one block for regex processing
    icd_block = ' '.join(icd_lines)

    # Remove group headers and their category names
    for header in group_headers:
        icd_block = icd_block.replace(header, ' ')

    category_names = [
        'Transport', 'Upper arm', 'Joint', 'Cancer', 'Heart',
        'Neurodevelopmental', 'Endocrinal and Gastro', 'Brain and Nervous System',
        'Health Service', 'Complications', 'Eye'
    ]
    for cat in category_names:
        icd_block = icd_block.replace(cat, ' ')

    # Now extract ICD entries with condition types
    # Pattern: ICD_CODE - Description ConditionType
    icd_pattern = re.compile(
        r'([A-Z]\d{1,2}(?:\.\d{1,4})?(?:[A-Z]{0,4}\d{0,2})?)\s*[-–]\s*'
        r'(.+?)\s+'
        r'(Cause|Accident|Pre-existing\s*Condition|Comorbidity)',
        re.IGNORECASE
    )

    for match in icd_pattern.finditer(icd_block):
        icd_code = match.group(1).strip()
        description = match.group(2).strip().rstrip(',')
        description = re.sub(r'\s+', ' ', description)
        condition_type = match.group(3).strip().lower()

        entry = f"{icd_code} - {description}"

        if condition_type == "cause":
            result["cause"].append(entry)
        elif condition_type == "accident":
            result["accident-related"].append(entry)
        elif "pre-existing" in condition_type:
            result["pre-existing condition"].append(entry)
        elif condition_type == "comorbidity":
            result["comorbidity"].append(entry)

    return result


def extract_icd_groups(pdf_path):
    """
    Main function to extract ICD groups from a medical summary PDF.

    Args:
        pdf_path: Path to the PDF file

    Returns:
        dict: JSON-compatible dictionary with categorized ICD codes
    """

    text = None
    extraction_method = None

    # Try pdfplumber first
    try:
        import pdfplumber
        text = extract_from_pdf_pdfplumber(pdf_path)
        extraction_method = "pdfplumber"
        print(f"Successfully extracted text using {extraction_method}")
    except ImportError:
        print("pdfplumber not installed. Trying PyMuPDF...")
    except Exception as e:
        print(f"pdfplumber failed: {e}. Trying PyMuPDF...")

    # Try PyMuPDF
    if not text:
        try:
            text = extract_from_pdf_pymupdf(pdf_path)
            extraction_method = "PyMuPDF"
            print(f"Successfully extracted text using {extraction_method}")
        except ImportError:
            print("PyMuPDF not installed. Trying pdfminer...")
        except Exception as e:
            print(f"PyMuPDF failed: {e}. Trying pdfminer...")

    # Try pdfminer
    if not text:
        try:
            text = extract_from_pdf_pdfminer(pdf_path)
            extraction_method = "pdfminer"
            print(f"Successfully extracted text using {extraction_method}")
        except ImportError:
            print("pdfminer not installed.")
        except Exception as e:
            print(f"pdfminer failed: {e}")

    if not text:
        print("ERROR: Could not extract text from PDF. Please install one of:")
        print("  pip install pdfplumber")
        print("  pip install PyMuPDF")
        print("  pip install pdfminer.six")
        return None

    # Try the structured parser first
    result = parse_icd_from_structured_text(text)

    # If structured parser didn't find much, try the line-by-line parser
    if result is None or sum(len(v) for v in result.values()) == 0:
        result = extract_icd_groups_from_text(text)

    # Remove duplicates while preserving order
    for key in result:
        seen = set()
        unique = []
        for item in result[key]:
            if item not in seen:
                seen.add(item)
                unique.append(item)
        result[key] = unique

    return result


def main():
    import sys
    import os

    # Default PDF path
    pdf_path = "./fake_files/GP-File-2-chronological_medical_summary_file.pdf"

    # # Allow command line argument for PDF path
    # if len(sys.argv) > 1:
    #     pdf_path = sys.argv[1]

    # if not os.path.exists(pdf_path):
    #     print(f"PDF file not found: {pdf_path}")
    #     print("Usage: python solution.py <path_to_pdf>")
    #     print()
    #     print("Since the PDF is not available, here's the expected output based on the document content:")
    #     print()

    #     # Hardcoded fallback based on the document content provided
    #     result = {
    #         "cause": [
    #             "V59.9XXD - Motor Vehicle Accident Injuring Restrained Driver, Subsequent Encounter",
    #             "V89.2XXA - Motor Vehicle Collision Injury",
    #             "V87 - Motor Vehicle Collison",
    #             "V89.2XXA - Person Injured In Unspecified Motor-Vehicle Accident, Traffic, Initial Encounter",
    #             "V89.2XXA - Motor Vehicular Accident",
    #             "V89.2XXA - Person Injured In Unspecified Motor-Vehicle Accident, Traffic, Subsequent Encounter",
    #             "V49.9XXA - Motor Vehicle Accident Injuring Restrained Driver, Initial Encounter",
    #             "V49.9XXA - Driver Injured In Collision With Unspecified Motor Vehicles In Ao Traffic Accident, Initial Encounter",
    #             "V49.40XA - Driver Injured In Collision With Unspecified Motor Vehicles In Traffic Accident, Initial Encounter",
    #             "V49.9XXA - Driver Injured In Collision With Unspecified Motor Vehicles In Ao Traffic Accident",
    #             "V89 - Dt Mva",
    #             "V89.2XXA - Motor Vehicle Accident",
    #             "V49.9XXA - Driver Injured In Collision"
    #         ],
    #         "accident-related": [
    #             "S42.002A - Left Clavicle Injury",
    #             "M25.512 - Pain",
    #             "M25 - Pain In Left Shoulder",
    #             "M25.512 - Pain In Left Shoulder",
    #             "Z04 - Accident",
    #             "T85 - Electrical Stimulation"
    #         ],
    #         "pre-existing condition": [
    #             "D49 - Brain Tumor",
    #             "D49.6 - Brain Tumor",
    #             "I63.9 - Stroke",
    #             "I10 - Hypertensive",
    #             "I63 - Cva",
    #             "I10 - Hypertension",
    #             "I69 - Stroke",
    #             "I60 - Hc",
    #             "F33 - Mdd",
    #             "F33.1 - Major Depressive Disorder",
    #             "F32 - Mdd",
    #             "F40 - Claustrophobic",
    #             "E29 - Testicular Hypofunction",
    #             "E66 - Obesity",
    #             "G47.33 - Obstructive Sleep Apnea",
    #             "G47 - Obstructive Sleep Apnea Of Fluid",
    #             "Z98.2 - Cerebrospinal Fluid Drainage Device",
    #             "T84 - Mrl Joint Upr Extrem W/O Dye",
    #             "H54 - Blind"
    #         ],
    #         "comorbidity": [
    #             "I10 - High Blood Pressure"
    #         ]
    #     }

    #     print(json.dumps(result, indent=2))

    #     # Save to file
    #     with open("icd_groups_output.json", "w") as f:
    #         json.dump(result, f, indent=2)
    #     print(f"\nOutput saved to icd_groups_output.json")

    #     return result

    # Extract from actual PDF
    result = extract_icd_groups(pdf_path)

    if result:
        print("\n" + "=" * 60)
        print("EXTRACTED ICD GROUPS")
        print("=" * 60)
        print(json.dumps(result, indent=2))

        # Save to file
        output_file = "icd_groups_output.json"
        with open(output_file, "w") as f:
            json.dump(result, f, indent=2)
        print(f"\nOutput saved to {output_file}")

        # Print summary
        print(f"\nSummary:")
        for key, values in result.items():
            print(f"  {key}: {len(values)} entries")
    else:
        print("Failed to extract ICD groups from the PDF.")

    return result


if __name__ == "__main__":
    main()

Successfully extracted text using pdfplumber

EXTRACTED ICD GROUPS
{
  "cause": [
    "V49.9XXA - Driver Injured In Collision"
  ],
  "accident-related": [
    "V59.9XXD - Motor Vehicle",
    "V89.2XXA - Motor Vehicle Collision Injury V87 - Motor Vehicle Collison V89.2XXA - Person Injured In Unspecified Motor-Vehicle",
    "V89.2XXA - Motor Vehicular",
    "V89.2XXA - Person Injured In Unspecified Motor-Vehicle",
    "V49.9XXA - Motor Vehicle",
    "V49.9XXA - Driver Injured In Collision With Unspecified Motor Vehicles In Ao Traffic",
    "V49.40XA - Driver Injured In Collision With Unspecified Motor Vehicles In Traffic",
    "V89 - Dt Mva V89.2XXA - Motor Vehicle",
    "S42.002A - Left Clavicle Injury",
    "M25.512 - Pain M25 - Pain In Left Shoulder M25.512 - Pain In Left Shoulder",
    "Z04 - Accident",
    "T85 - Electrical Stimulation"
  ],
  "pre-existing condition": [
    "D49 - Brain Tumor D49.6 - Brain Tumor",
    "I69 - Stroke",
    "I60 - Hc",
    "F33 - Mdd",
    "F33.1 - M

In [46]:
def extract_medical_conditions_from_medical_summary(medical_entities):
    """
    Extract and categorize medical conditions from the medical summary entities.
    
    Args:
        medical_entities: List of medical entities extracted from the summary, 
                        each with 'text' and 'label' (e.g., 'Condition', 'Medication')
    """
    final_json = {}
    final_json["Cause"] = []
    final_json["Accident"] = []
    final_json["Pre-existing Condition"] = []
    final_json["Comorbidity"] = []
    patients_data = medical_entities.get("patients", [])
    for patient_data in patients_data:
        medical_entities = patient_data.get("medical_history", [])
        for medical_condition in medical_entities:
            condition_name = medical_condition.get("condition_type", "jklkjl")
            if not condition_name:
                continue
            condition_string = f"{medical_condition.get('icd10cm_concept_code', '')} - {medical_condition.get('answer', '')}"
            if "Cause" in condition_name:
                final_json["Cause"].append(condition_string)
            elif "Accident" in condition_name:
                final_json["Accident"].append(condition_string)
            elif "Pre-existing Condition" in condition_name:
                final_json["Pre-existing Condition"].append(condition_string)
            elif "Comorbidity" in condition_name:
                final_json["Comorbidity"].append(condition_string)

    return final_json

In [50]:
import json

medical_entities_path = "./fake_files/medical_entities.json"
with open(medical_entities_path, "r") as f:
    medical_entities = json.load(f)

result = extract_medical_conditions_from_medical_summary(medical_entities)
print(json.dumps(result, indent=2))

{
  "Cause": [
    "V89 - DT MVA",
    "V87 - motor vehicle collison",
    "V49.9XXA - Driver injured in collision with unspecified motor vehicles in ao\ntraffic accident",
    "V49.9XXA - Driver injured in collision with unspecified motor vehicles in ao\ntraffic accident, initial encounter",
    "V89.2XXA - Motor Vehicle Collision Injury",
    "V49.40XA - Driver injured in collision with unspecified motor vehicles in traffic accident, initial encounter",
    "V89.2XXA - Motor Vehicle Collision Injury",
    "V89.2XXA - Collision Injury",
    "V89 - MVA",
    "V89 - MVA",
    "V49.9XXA - Driver injured in collision",
    "V49.40XA - Driver injured in collision with unspecified motor vehicles in traffic accident, initial encounter",
    "V89 - MVC",
    "V89.2XXA - Person injured in unspecified motor-vehicle accident, traffic, subsequent encounter",
    "V89.2XXA - Person injured in unspecified motor-vehicle accident, traffic, initial encounter",
    "V89.2XXA - motor vehicle accident",


In [26]:
import json

medical_entities_path = "./fake_files/medical_entities.json"
with open(medical_entities_path, "r") as f:
    medical_entities = json.load(f)

final_json = {}
final_json["Cause"] = []
final_json["Accident"] = []
final_json["Pre-existing Condition"] = []
final_json["Comorbidity"] = []

patients_data = medical_entities.get("patients", [])
for patient_data in patients_data:
    medical_entities = patient_data.get("medical_history", [])
    for medical_condition in medical_entities:
        condition_name = medical_condition.get("condition_type", "jklkjl")
        if not condition_name:
            continue
        condition_string = f"{medical_condition.get('icd10cm_concept_code', '')} - {medical_condition.get('answer', '')}"
        if "Cause" in condition_name:
            final_json["Cause"].append(condition_string)
        elif "Accident" in condition_name:
            final_json["Accident"].append(condition_string)
        elif "Pre-existing Condition" in condition_name:
            final_json["Pre-existing Condition"].append(condition_string)
        elif "Comorbidity" in condition_name:
            final_json["Comorbidity"].append(condition_string)

print(final_json)
final_json_path = "./fake_files/final_icd_groups.json"
with open(final_json_path, "w") as f:
    json.dump(final_json, f, indent=2)
print(f"Final JSON saved to: {final_json_path}")

{'Cause': ['V89 - DT MVA', 'V87 - motor vehicle collison', 'V49.9XXA - Driver injured in collision with unspecified motor vehicles in ao\ntraffic accident', 'V49.9XXA - Driver injured in collision with unspecified motor vehicles in ao\ntraffic accident, initial encounter', 'V89.2XXA - Motor Vehicle Collision Injury', 'V49.40XA - Driver injured in collision with unspecified motor vehicles in traffic accident, initial encounter', 'V89.2XXA - Motor Vehicle Collision Injury', 'V89.2XXA - Collision Injury', 'V89 - MVA', 'V89 - MVA', 'V49.9XXA - Driver injured in collision', 'V49.40XA - Driver injured in collision with unspecified motor vehicles in traffic accident, initial encounter', 'V89 - MVC', 'V89.2XXA - Person injured in unspecified motor-vehicle accident, traffic, subsequent encounter', 'V89.2XXA - Person injured in unspecified motor-vehicle accident, traffic, initial encounter', 'V89.2XXA - motor vehicle accident', 'V89.2XXA - Motor vehicle accident', 'V49.9XXA - Motor vehicle accid

In [29]:
import pdfplumber
import json
import re
import sys
import os


def clean_text(text):
    """Remove citation markers like [1][2][3] from text."""
    cleaned = re.sub(r'$$\d+$$', '', text)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


def extract_list_items(text_block):
    """Extract numbered list items from a text block."""
    lines = text_block.split('\n')
    items = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Match lines starting with a number and period (e.g., "1.", "2.")
        match = re.match(r'^\d+\.\s+(.+)', line)
        if match:
            items.append(clean_text(match.group(1)))
    return items


def extract_damages_injuries(pdf_path):
    """
    Extract specific fields from the Damages/Injuries section of a PDF.

    Args:
        pdf_path (str): Path to the input PDF file.

    Returns:
        dict: Extracted data as a dictionary.
    """
    full_text = ""

    # Extract all text from the PDF
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                full_text += page_text + "\n"

    # -------------------------------------------------------------------------
    # Locate the Damages/Injuries section
    # -------------------------------------------------------------------------
    damages_pattern = re.search(
        r'3\.\s*Damages/Injuries(.*?)(?=4\.\s*Economic Damages Claimed|$)',
        full_text,
        re.DOTALL | re.IGNORECASE
    )

    if not damages_pattern:
        print("Warning: 'Damages/Injuries' section not found in the PDF.")
        damages_text = full_text  # Fall back to full text
    else:
        damages_text = damages_pattern.group(1)

    # -------------------------------------------------------------------------
    # 1. Injuries/Medical Conditions Identified
    # -------------------------------------------------------------------------
    injuries_identified = []

    injuries_identified_match = re.search(
        r'Injuries/Medical Conditions Identified\s*(.*?)'
        r'(?=Injuries/Medical conditions Related to Accident|$)',
        damages_text,
        re.DOTALL | re.IGNORECASE
    )

    if injuries_identified_match:
        block = injuries_identified_match.group(1)
        items = extract_list_items(block)
        if items:
            injuries_identified = items
        else:
            # If no numbered list found, capture the raw cleaned text
            injuries_identified = [clean_text(block)]

    # -------------------------------------------------------------------------
    # 2. Injuries/Medical Conditions Related to Accident or Incident
    # -------------------------------------------------------------------------
    injuries_related = ""

    injuries_related_match = re.search(
        r'Injuries/Medical conditions Related to Accident or Incident\s*(.*?)'
        r'(?=Medical conditions or injuries that existed prior|$)',
        damages_text,
        re.DOTALL | re.IGNORECASE
    )

    if injuries_related_match:
        block = injuries_related_match.group(1)
        injuries_related = clean_text(block)

    # -------------------------------------------------------------------------
    # 3. Medical Conditions or Injuries Prior to Accident / Not Related
    # -------------------------------------------------------------------------
    pre_existing = []

    pre_existing_match = re.search(
        r'Medical conditions or injuries that existed prior to the accident or incident'
        r'.*?accident/incident\s*(.*?)'
        r'(?=Medical providers that have examined|$)',
        damages_text,
        re.DOTALL | re.IGNORECASE
    )

    if pre_existing_match:
        block = pre_existing_match.group(1)
        items = extract_list_items(block)
        if items:
            pre_existing = items
        else:
            pre_existing = [clean_text(block)]

    # -------------------------------------------------------------------------
    # Build output dictionary
    # -------------------------------------------------------------------------
    output = {
        "Injuries/Medical Conditions Identified": injuries_identified,
        "Injuries/Medical Conditions Related to Accident or Incident and Confirmed by Medical Providers": injuries_related,
        "Medical conditions or injuries that existed prior to the accident or incident "
        "or are not related to the accident/incident": pre_existing
    }

    return output


def main():
    # -------------------------------------------------------------------------
    # Handle command-line arguments
    # -------------------------------------------------------------------------
    # if len(sys.argv) < 2:
    #     print("Usage: python solution.py <input_pdf_path> [output_json_path]")
    #     print("Example: python solution.py GP-File-2-claim_summary.pdf output.json")
    #     sys.exit(1)

    # pdf_path = sys.argv[1]

    # # Default output JSON path: same name as PDF but with .json extension
    # if len(sys.argv) >= 3:
    #     output_json_path = sys.argv[2]
    # else:
    #     base_name = os.path.splitext(pdf_path)[0]
    #     output_json_path = base_name + "_damages_injuries.json"

    # # -------------------------------------------------------------------------
    # # Validate input file
    # # -------------------------------------------------------------------------
    # if not os.path.exists(pdf_path):
    #     print(f"Error: File '{pdf_path}' not found.")
    #     sys.exit(1)

    # if not pdf_path.lower().endswith('.pdf'):
    #     print("Warning: Input file may not be a PDF.")

    # -------------------------------------------------------------------------
    # Extract data and write JSON
    # -------------------------------------------------------------------------
    pdf_path = "./fake_files/GP-File-2-claim_summary.pdf"  # Override for testing without command-line args
    base_name = os.path.splitext(pdf_path)[0]
    output_json_path = base_name + "_damages_injuries.json"
    print(f"Processing: {pdf_path}")
    data = extract_damages_injuries(pdf_path)

    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Output saved to: {output_json_path}")
    print("\nExtracted Data:")
    print(json.dumps(data, indent=4))


if __name__ == "__main__":
    main()

Processing: ./fake_files/GP-File-2-claim_summary.pdf
Output saved to: ./fake_files/GP-File-2-claim_summary_damages_injuries.json

Extracted Data:
{
    "Injuries/Medical Conditions Identified": [
        "Acute pain in left shoulder",
        "Motor vehicle accident injury (as a restrained driver)",
        "Unspecified motor vehicle accident-related injury"
    ],
    "Injuries/Medical Conditions Related to Accident or Incident and Confirmed by Medical Providers": "Medical provider statements confirming accident-related injuries for Roger Pouncey: The documents do not include a statement by a healthcare or medical provider confirming that the injuries or medical conditions of Roger Pouncey were caused by the accident or incident.[1][3][20][7][11][5][40][19][2][21]",
    "Medical conditions or injuries that existed prior to the accident or incident or are not related to the accident/incident": [
        "Stroke affecting right upper extremity",
        "Brain tumor from childhood (at a

In [34]:
import pandas as pd
import numpy as np 
import json
import boto3 
import re

def extract_and_eval_json(text: str):
    """
    Extracts a substring that starts with { and ends with } from the given text
    and evaluates it as JSON.
    """
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return eval(match.group())
        except json.JSONDecodeError as e:
            print(f"JSON decoding error: {e}")
            return {}
    return {}

def get_output_from_sonnet(bedrock_client, prompt):
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 4096,
        "messages": [
            {
                "role": "user",
                "content": prompt,
            }
        ],
    }

    response = bedrock_client.invoke_model(
        modelId="us.anthropic.claude-sonnet-4-20250514-v1:0",
        body=json.dumps(request_body),
        contentType="application/json",
        accept="application/json",
    )

    response_body = json.loads(response["body"].read())
    response_text = response_body["content"][0]["text"]
    return response_text


session = boto3.Session(profile_name = 'aidev', region_name = 'us-east-1')
bedrock_client = session.client(service_name = 'bedrock-runtime', region_name = 'us-east-1')

with open("./fake_files/final_icd_groups.json", "r") as f:
    medical_summary_output = json.load(f)

with open("./fake_files/GP-File-2-claim_summary_damages_injuries.json", "r") as f:
    claim_summary_output = json.load(f)

data = pd.read_csv("cache/dataframe/ae52691527d1858df32f93975ceaa2be.csv")
bill_items = data['Medical Procedure Name'].unique()

prompt = f"""
You are an expert medical bill reviewer. You will be given two summarized data. One is medical summary output which has the conditions categorized by "Cause", "Accident", "Pre-existing Condition", and "Comorbidity". The other is claim summary output which has the injuries/medical conditions identified, injuries/medical conditions related to accident or incident and confirmed by the medical providers, and medical conditions or injuries prior to accident or not related to accident.
Then you will be given all the medical procedures and drug items billed in the medical bill. Now, your task is to associate the necessity of each of the bill items and categorize strictly into the following three categories:
- "Necessary and Medically stated": The bill item is necessary (accident related) and also confirmed by the medical providers in the claim summary output.
- "Necessary but Not Medically stated": The bill item is necessary (accident related) but not yet confirmed by the medical providers in the claim summary output.
- "Not necessary": The bill item is not necessary (not accident related). It might be pre-existing conditions or comorbidity which have not been affected after the accident.

The output should strictly be a JSON output with the bill items as keys and the values as a json containing the category key which is one of the above 3, and the reason key which is a very brief reason explaining the categorization.
Here are the input information:
Medical Summary Output: {medical_summary_output}
Claim Summary Output: {claim_summary_output}
Bill Items: {bill_items}

Output:
"""

response_text = get_output_from_sonnet(bedrock_client, prompt)
json_output = extract_and_eval_json(response_text)

print(json_output)

{'XR CLAVICLE LT': {'category': 'Necessary but Not Medically stated', 'reason': 'Left clavicle injury identified in accident conditions but no provider confirmation of accident causation'}, 'XR SHOULDER 2V/MORE': {'category': 'Necessary but Not Medically stated', 'reason': 'Left shoulder pain identified in accident conditions but no provider confirmation of accident causation'}, 'DEXAMETHASONE 4MG/ML': {'category': 'Necessary but Not Medically stated', 'reason': 'Anti-inflammatory for accident-related shoulder/clavicle injuries but no provider confirmation'}, 'KETOROLAC 60MG VIAL': {'category': 'Necessary but Not Medically stated', 'reason': 'Pain medication for accident-related injuries but no provider confirmation of accident causation'}, 'METHOCARBAMOL 500MG': {'category': 'Necessary but Not Medically stated', 'reason': 'Muscle relaxant for accident-related injuries but no provider confirmation of accident causation'}, 'ER LEVEL FOUR - 25': {'category': 'Necessary but Not Medically 

In [35]:
with open("./fake_files/bill_review_output.json", "w") as f:
    json.dump(json_output, f, indent=4)
print("Bill review output saved to: ./fake_files/bill_review_output.json")

Bill review output saved to: ./fake_files/bill_review_output.json


In [36]:
data.head()

,Document Name,Date,Medical Procedure Name,CPT Code,CPT Code Source,Modifier,Quantity,Unit Price,Total Price,Entity Type,...,Dosage Unit,Units from Name,Page,Extraction Confidence,Unit Price Threshold,No. of Medicare Units,Total Threshold Price,Threshold Multiplier,Price Difference,Highlight Geometry
0,GP2.pdf,2022-08-05,XR CLAVICLE LT,73000,extracted,LT,1,200.00,200.00,procedure,...,NaN,NaN,3,95,25.38,1.0,25.38,7.88,174.62,"{""Width"": 0.355433851480484, ""Height"": 0.01183..."
1,GP2.pdf,2022-08-05,XR SHOULDER 2V/MORE,73030,extracted,LT,1,220.00,220.00,procedure,...,NaN,NaN,3,95,26.72,1.0,26.72,8.23,193.28,"{""Width"": 0.25812920928001404, ""Height"": 0.011..."
2,GP2.pdf,2022-08-05,DEXAMETHASONE 4MG/ML,J1100,inferred,NaN,4,7.92,31.68,drug,...,MG,1.0,3,90,0.10,16.0,1.68,18.86,30.00,"{""Width"": 0.2683259844779968, ""Height"": 0.0114..."
3,GP2.pdf,2022-08-05,KETOROLAC 60MG VIAL,J1885,inferred,NaN,4,12.00,48.00,drug,...,MG,1.0,3,90,0.33,16.0,5.33,9.01,42.67,"{""Width"": 0.2581912577152252, ""Height"": 0.0096..."
4,GP2.pdf,2022-08-05,METHOCARBAMOL 500MG,J2800,inferred,NaN,2,6.00,12.00,drug,...,MG,1.0,3,85,4.54,2.0,9.07,1.32,2.93,"{""Width"": 0.25746843218803406, ""Height"": 0.009..."


In [37]:
data["Necessity Flag"] = data["Medical Procedure Name"].apply(lambda x: json_output.get(x, {}).get("category", "Not categorized"))
data["Necessity Reason"] = data["Medical Procedure Name"].apply(lambda x: json_output.get(x, {}).get("reason", "No reason provided"))

In [38]:
data.head()

,Document Name,Date,Medical Procedure Name,CPT Code,CPT Code Source,Modifier,Quantity,Unit Price,Total Price,Entity Type,...,Page,Extraction Confidence,Unit Price Threshold,No. of Medicare Units,Total Threshold Price,Threshold Multiplier,Price Difference,Highlight Geometry,Necessity Flag,Necessity Reason
0,GP2.pdf,2022-08-05,XR CLAVICLE LT,73000,extracted,LT,1,200.00,200.00,procedure,...,3,95,25.38,1.0,25.38,7.88,174.62,"{""Width"": 0.355433851480484, ""Height"": 0.01183...",Necessary but Not Medically stated,Left clavicle injury identified in accident co...
1,GP2.pdf,2022-08-05,XR SHOULDER 2V/MORE,73030,extracted,LT,1,220.00,220.00,procedure,...,3,95,26.72,1.0,26.72,8.23,193.28,"{""Width"": 0.25812920928001404, ""Height"": 0.011...",Necessary but Not Medically stated,Left shoulder pain identified in accident cond...
2,GP2.pdf,2022-08-05,DEXAMETHASONE 4MG/ML,J1100,inferred,NaN,4,7.92,31.68,drug,...,3,90,0.10,16.0,1.68,18.86,30.00,"{""Width"": 0.2683259844779968, ""Height"": 0.0114...",Necessary but Not Medically stated,Anti-inflammatory for accident-related shoulde...
3,GP2.pdf,2022-08-05,KETOROLAC 60MG VIAL,J1885,inferred,NaN,4,12.00,48.00,drug,...,3,90,0.33,16.0,5.33,9.01,42.67,"{""Width"": 0.2581912577152252, ""Height"": 0.0096...",Necessary but Not Medically stated,Pain medication for accident-related injuries ...
4,GP2.pdf,2022-08-05,METHOCARBAMOL 500MG,J2800,inferred,NaN,2,6.00,12.00,drug,...,3,85,4.54,2.0,9.07,1.32,2.93,"{""Width"": 0.25746843218803406, ""Height"": 0.009...",Necessary but Not Medically stated,Muscle relaxant for accident-related injuries ...


In [40]:
data["Necessity Reason"].unique()

array(['Left clavicle injury identified in accident conditions but no provider confirmation of accident causation',
       'Left shoulder pain identified in accident conditions but no provider confirmation of accident causation',
       'Anti-inflammatory for accident-related shoulder/clavicle injuries but no provider confirmation',
       'Pain medication for accident-related injuries but no provider confirmation of accident causation',
       'Muscle relaxant for accident-related injuries but no provider confirmation of accident causation',
       'Emergency care following motor vehicle accident but no provider confirmation of accident causation',
       'Administration of accident-related medications but no provider confirmation of accident causation',
       'Post-accident evaluation for identified injuries but no provider confirmation of accident causation',
       'Listed as pre-existing condition and related to prior stroke affecting right upper extremity',
       'Transport fol

In [41]:
flag_map = {
    "necessary but stated": "🟢",
    "necessary but not stated": "🟡",
    "unnecessary": "🔴",
}

data["Necessity Status"] = data["Necessity Flag"].map(flag_map)

# Move Status column to the front
cols = ["Necessity Status"] + [c for c in data.columns if c != "Necessity Status"]
data = data[cols]

In [42]:
data.head()

,Necessity Status,Document Name,Date,Medical Procedure Name,CPT Code,CPT Code Source,Modifier,Quantity,Unit Price,Total Price,...,Page,Extraction Confidence,Unit Price Threshold,No. of Medicare Units,Total Threshold Price,Threshold Multiplier,Price Difference,Highlight Geometry,Necessity Flag,Necessity Reason
0,NaN,GP2.pdf,2022-08-05,XR CLAVICLE LT,73000,extracted,LT,1,200.00,200.00,...,3,95,25.38,1.0,25.38,7.88,174.62,"{""Width"": 0.355433851480484, ""Height"": 0.01183...",Necessary but Not Medically stated,Left clavicle injury identified in accident co...
1,NaN,GP2.pdf,2022-08-05,XR SHOULDER 2V/MORE,73030,extracted,LT,1,220.00,220.00,...,3,95,26.72,1.0,26.72,8.23,193.28,"{""Width"": 0.25812920928001404, ""Height"": 0.011...",Necessary but Not Medically stated,Left shoulder pain identified in accident cond...
2,NaN,GP2.pdf,2022-08-05,DEXAMETHASONE 4MG/ML,J1100,inferred,NaN,4,7.92,31.68,...,3,90,0.10,16.0,1.68,18.86,30.00,"{""Width"": 0.2683259844779968, ""Height"": 0.0114...",Necessary but Not Medically stated,Anti-inflammatory for accident-related shoulde...
3,NaN,GP2.pdf,2022-08-05,KETOROLAC 60MG VIAL,J1885,inferred,NaN,4,12.00,48.00,...,3,90,0.33,16.0,5.33,9.01,42.67,"{""Width"": 0.2581912577152252, ""Height"": 0.0096...",Necessary but Not Medically stated,Pain medication for accident-related injuries ...
4,NaN,GP2.pdf,2022-08-05,METHOCARBAMOL 500MG,J2800,inferred,NaN,2,6.00,12.00,...,3,85,4.54,2.0,9.07,1.32,2.93,"{""Width"": 0.25746843218803406, ""Height"": 0.009...",Necessary but Not Medically stated,Muscle relaxant for accident-related injuries ...


In [43]:
data.to_csv("cache/dataframe/ae52691527d1858df32f93975ceaa2be.csv", index=False)

In [44]:
data.drop(columns=["Necessity Status"], inplace=True)

In [45]:
data.to_csv("cache/dataframe/ae52691527d1858df32f93975ceaa2be.csv", index=False)